# 04 — Error decomposition: *why* are the distributions wrong?

SimBench gives one number per item, `S = 100·(1 − TVD(P,Q)/Z)`. That says *how
much* a prediction is off, not *how*. This notebook takes **one run** and
compares the predicted distributions against ground truth, splitting the error
into interpretable parts.

**The decomposition.** For truth `P` and prediction `Q`, sort both descending and
split the total-variation distance:

```
TVD(P,Q) = concentration_err + location_err
         = ½Σ|sort(P)−sort(Q)|  +  (TVD − that)
```

- **concentration_err** — error in the *profile* (how peaked vs spread the mass
  is), independent of *which* option carries it. *Are we capturing the group's
  true diversity vs concentration?*
- **location_err** — extra error from putting otherwise well-shaped mass on the
  *wrong options*. *Is the weight in the right place?*

Both ≥ 0 and they sum to TVD. Since the normalizer `Z` depends only on the truth,
a **score gap between two systems splits the same way** — so we can attribute a
score *gain* to "fixed the spread" vs "moved mass onto the right options".

Signed facets add direction: **entropy_gap** = H(Q)−H(P) (`<0` over-sharpened,
`>0` too diffuse), **mode_acc** (did we pick the truth's dominant option).


In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# make src importable when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent / "src"))

from scrye.config import OUTPUTS_DIR
from scrye.data import load_all
from scrye.splits import make_split
from scrye.evaluate import build_normalizers
from scrye.decompose import decompose_records, decompose_error, system_summary

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# --- pick a run -------------------------------------------------------------
# Any outputs/runs/<RUN>.results.json works. The model-sweep is richest: it holds
# the full progression (gemini 2.5 / 3.1 / 3.5 x faithful / anti_flattening /
# contextualized), so the "why did we get better" story is visible in one run.
RUN = "2026-06-20-model-sweep"
results_path = OUTPUTS_DIR / "runs" / f"{RUN}.results.json"
per_system = json.load(open(results_path))
print(f"run: {RUN}\nsystems ({len(per_system)}):")
for k in per_system:
    print("  ", k)


## Normalizers + decomposed frames

`Z` (SimBench Eq. 2) is built from the **full split**, exactly as every run
script does, so the score-scale loss columns line up with the recorded scores.
We then decompose every record of every system.


In [ ]:
allr = load_all()
full = allr["grouped"] + allr["pop"]
split = make_split(full, seed=0, unit="question")
normalizers = build_normalizers(split.dev + split.val + split.test)

# one decomposed frame per system; tag with the system label
frames = {}
for sysname, recs in per_system.items():
    df = decompose_records(recs, normalizers=normalizers)
    df["system"] = sysname
    frames[sysname] = df

all_df = pd.concat(frames.values(), ignore_index=True)
print(f"decomposed {len(all_df)} (record x system) rows")
all_df[["system", "split", "dataset", "tvd", "concentration_err",
        "location_err", "entropy_gap", "mode_match", "score"]].head()


## Headline: per-system error profile (grouped split)

`grouped` is where demographic conditioning actually operates (pop is
near-unconditioned). `score_loss = 100 − score`, split into the part from
**concentration** (wrong spread) and **location** (wrong options). Lower is
better; the split shows *which* kind of error each system pays.


In [ ]:
def summary_table(df, split="grouped"):
    rows = []
    for sysname, f in df.groupby("system"):
        s = system_summary(f[f["split"] == split])
        s.name = sysname
        rows.append(s)
    return pd.DataFrame(rows)

grouped_tbl = summary_table(all_df, "grouped").sort_values("score_loss")
cols = ["n", "score_loss", "conc_loss", "loc_loss", "conc_share",
        "entropy_gap", "mode_acc", "mode_mass_err"]
display(grouped_tbl[cols].round(3))

print("\nReading:")
print("  conc_share ~0.65-0.89 -> most error is PROFILE/spread, not wrong-options")
print("  entropy_gap > 0        -> predictions are MORE diffuse than truth (too flat)")


## The two wins, decomposed

The method search produced two real gains. The stacked bars show what each gain
actually *fixed* — concentration (spread) vs location (options).


In [ ]:
def loss_split_bar(tbl, title):
    t = tbl.sort_values("score_loss", ascending=True)
    fig, ax = plt.subplots(figsize=(9, 0.6 * len(t) + 1.5))
    y = np.arange(len(t))
    ax.barh(y, t["conc_loss"], color="#4C72B0", label="concentration (wrong spread)")
    ax.barh(y, t["loc_loss"], left=t["conc_loss"], color="#DD8452",
            label="location (wrong options)")
    ax.set_yticks(y); ax.set_yticklabels(t.index)
    ax.invert_yaxis()
    ax.set_xlabel("score loss below 100  (= 100 - SimBench S)")
    ax.set_title(title)
    ax.legend(loc="lower right", fontsize=8)
    for yi, (c, l) in enumerate(zip(t["conc_loss"], t["loc_loss"])):
        ax.annotate(f"{c+l:.0f}", (c + l, yi), xytext=(3, 0),
                    textcoords="offset points", va="center", fontsize=8)
    plt.tight_layout(); plt.show()

loss_split_bar(grouped_tbl, "Grouped score loss = concentration + location")


In [ ]:
# Isolate the two transitions, if this run has them (model-sweep does).
def get(sys_label):
    return grouped_tbl.loc[sys_label] if sys_label in grouped_tbl.index else None

pairs = [
    ("Model upgrade (faithful)", "gemini-flash-lite::faithful", "gemini-3.1-flash-lite::faithful"),
    ("Strategy (faithful->anti_flattening @3.1)", "gemini-3.1-flash-lite::faithful", "gemini-3.1-flash-lite::anti_flattening"),
]
print(f"{'transition':46s} {'dScore':>8} {'dConc':>8} {'dLoc':>8}  attribution")
for label, a, b in pairs:
    ra, rb = get(a), get(b)
    if ra is None or rb is None:
        continue
    dscore = ra["score_loss"] - rb["score_loss"]     # positive = improvement
    dconc = ra["conc_loss"] - rb["conc_loss"]
    dloc = ra["loc_loss"] - rb["loc_loss"]
    drv = "concentration" if abs(dconc) > abs(dloc) else "location"
    print(f"{label:46s} {dscore:+8.2f} {dconc:+8.2f} {dloc:+8.2f}  mostly {drv}")


## Distributional comparison: predicted vs true *spread*

Each point is one item: predicted normalized entropy vs the truth's. Points
**above** the diagonal = the system is too diffuse (over-spread); **below** =
too sharp. This is the concentration error made visual — and doubles as the
entropy-calibration diagnostic (is there a systematic, fixable bias, or just
noise around the line?).


In [ ]:
def entropy_scatter(systems, split="grouped"):
    n = len(systems)
    cols = min(3, n); rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 3.8 * rows), squeeze=False)
    for ax, sysname in zip(axes.flat, systems):
        f = frames[sysname]; f = f[f["split"] == split]
        ax.scatter(f["truth_entropy"], f["pred_entropy"], s=8, alpha=0.35,
                   color="#55A868")
        ax.plot([0, 1], [0, 1], "k--", lw=1)
        gap = f["entropy_gap"].mean()
        ax.set_title(f"{sysname}\nmean gap {gap:+.3f}", fontsize=8)
        ax.set_xlabel("truth entropy"); ax.set_ylabel("pred entropy")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    for ax in axes.flat[n:]:
        ax.axis("off")
    plt.tight_layout(); plt.show()

# show the three systems at the winning model
show = [s for s in per_system if s.startswith("gemini-3.1-flash-lite")] or list(per_system)[:3]
entropy_scatter(show)


## Where do the gains land? Error by truth-entropy regime

SimBench finds models do worst on high-entropy (contested) items. Binning items
into low / mid / high truth-entropy tertiles shows whether a system's advantage
comes from the easy consensus items or the hard contested ones.


In [ ]:
def by_entropy_regime(df, systems, split="grouped"):
    g = df[(df["split"] == split) & (df["system"].isin(systems))].copy()
    # tertiles on the pooled truth-entropy distribution (shared bin edges)
    edges = g["truth_entropy"].quantile([0, 1/3, 2/3, 1.0]).to_numpy(copy=True)
    edges[0] -= 1e-9
    g["regime"] = pd.cut(g["truth_entropy"], bins=edges,
                         labels=["low (consensus)", "mid", "high (contested)"])
    piv = g.groupby(["system", "regime"], observed=True).apply(
        lambda x: 100 - system_summary(x)["score_loss"]).unstack()
    return piv

regime_tbl = by_entropy_regime(all_df, show)
print("mean SimBench score by truth-entropy regime (grouped):")
display(regime_tbl.round(1))


## Side-by-side: predicted vs true distributions

Concrete items — the best-fit, a typical, and the worst-fit grouped item for the
chosen system — with each item's concentration/location split annotated.


In [ ]:
FOCUS = "gemini-3.1-flash-lite::anti_flattening"
FOCUS = FOCUS if FOCUS in frames else list(frames)[0]
recs = per_system[FOCUS]
fdf = frames[FOCUS]
g_idx = fdf.index[fdf["split"] == "grouped"].tolist()
gsub = fdf.loc[g_idx].sort_values("tvd")
picks = {"best fit": gsub.index[0],
         "typical": gsub.index[len(gsub) // 2],
         "worst fit": gsub.index[-1]}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (label, idx) in zip(axes, picks.items()):
    rec = recs[idx]
    opts = list(rec["truth"])
    truth = [rec["truth"].get(o, 0) for o in opts]
    pred = [rec["pred"].get(o, 0) for o in opts]
    x = np.arange(len(opts)); w = 0.4
    ax.bar(x - w/2, truth, w, label="truth", color="#4C72B0")
    ax.bar(x + w/2, pred, w, label="pred", color="#DD8452")
    row = fdf.loc[idx]
    ax.set_title(f"{label} | {rec['dataset']}\nconc {row['concentration_err']:.2f} / "
                 f"loc {row['location_err']:.2f}", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(opts, fontsize=7); ax.legend(fontsize=7)
fig.suptitle(f"{FOCUS} — predicted vs true (grouped)", fontsize=10)
plt.tight_layout(); plt.show()


## Pop vs grouped contrast

Pop is near-unconditioned, so the *location* axis has little to grab onto there;
grouped is where option-placement matters. Contrasting the split confirms the
conditioning gains are a grouped-split phenomenon.


In [ ]:
rows = []
for split in ("pop", "grouped"):
    t = summary_table(all_df, split)
    t["split"] = split
    rows.append(t[["split", "score_loss", "conc_loss", "loc_loss", "conc_share", "entropy_gap"]])
contrast = pd.concat(rows).reset_index().rename(columns={"index": "system"})
display(contrast.round(3).sort_values(["system", "split"]).set_index(["system", "split"]))


## What this run says

*(model-sweep; grouped split)*

1. **Most error is concentration, not location** (`conc_share` ≈ 0.65–0.89). We
   broadly rank the right options; we get the *shape* — how diversity vs
   concentration is distributed — wrong.
2. **The big lever (model 2.5→3.1, ≈ −17 score loss) fixed both, concentration-led**
   (≈ −10 conc, −7 loc): a better model both calibrates the spread *and* finds the
   right options more often (mode acc 0.51 → 0.58).
3. **The strategy win (anti_flattening) is purely location** (≈ −4 loc, ~0 conc).
   Despite the name, it does not improve the spread profile — it moves mass onto
   the *right* (often minority) options that the faithful prompt zeroed out.
4. **We are too diffuse on grouped** (`entropy_gap > 0`), not too sharp. That
   reframes calibration: "flatten more" was always going to fail (Stage 06's
   `TempScaling T>1` did), because the concentration error is per-item and
   bidirectional, not a global over-sharpening to temper away.
5. **3.5-flash loses on grouped by over-diffusing** — best option coverage (lowest
   location error) but worst concentration error, so net worse than 3.1.

The open headroom is concentration on contested (high-entropy) items, and it is
*not* monotone — which is why a single global/own-entropy calibrator couldn't
capture it.
